# 震智杯 · Colab 一键训练（超大数据集 · 90GB 盘友好）

**从上到下依次运行即可**。断连/重开机后：重新跑 Cell 1–3，然后直接跑你中断处的那个 Cell——
所有重活（取数、训练）都会**自动断点续传**（进度和产物在 Google Drive）。

核心策略（为什么 90GB 盘能吃 300GB 数据集）：
1. **基座模型直接用 `PhaseNet('diting')`**——USTC 用 271 万条中国 DiTing 数据训好的 picker，
   一行加载（几 MB），等于白拿整个 DiTing 的知识，**一个字节原始数据都不用下**；
2. 微调数据用 `chunked_fetch.py`：**下载 1 块 → 抽 3001 点训练窗 → 删块 → 下一块**，
   峰值磁盘 = 单块 + 紧凑池（10 万窗 ≈ 3.5GB），与数据集总大小无关；
3. 按**事件**切 train/holdout（防泄漏），best 权重以 holdout 官方计分守门。

| 磁盘预算 | 大小 |
|---|---|
| seisbench 单块缓存（用完即删） | 峰值 ~10–30GB |
| 紧凑训练池（8 万标注窗 + 1 万噪声窗） | ~3GB |
| Drive 落盘（池 + checkpoint） | ~3GB（免费 15GB 内） |

In [ ]:
# Cell 1 —— 挂载 Google Drive（产物持久化的根）
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/dizheng'
os.makedirs(DRIVE, exist_ok=True)
print('Drive 就绪:', DRIVE)

In [ ]:
# Cell 2 —— 参数（一般不用改）
REPO_URL   = 'https://github.com/ixijxjgxidj-cmd/dz.git'   # 你的仓库
REPO_DIR   = '/content/dz'
CACHE      = '/content/sb_cache'          # seisbench 块缓存（临时盘，块用完即删）
POOL       = '/content/pool.hdf5'         # 紧凑训练池（会镜像到 Drive）
PRETRAINED = 'diting'                      # 基座：USTC 中国数据 PhaseNet（比 stead 更贴赛题域）
SR         = 50                            # 必须=基座原生采样率：diting=50Hz、stead=100Hz（取数/训练全链路统一，finetune 脚本会校验）
WIN        = 3001                          # 训练窗长=PhaseNet in_samples（50Hz 下 60.02s）
DATASET    = 'CWA'                         # 标注大集：台湾 CWA（P/S 标注、按年分块；seisbench 重采样到 SR 时会同步换算到时下标）
NOISE_SET  = 'CWANoise'                    # 纯噪声块（赛题含噪声条目，练“不误报”）
N_LABELED  = 80000                         # 标注窗数上限（标准 12.7GB RAM 安全值；High-RAM 可到 15 万）
N_NOISE    = 10000
RUN_DIR    = f'{DRIVE}/runs/ft_{PRETRAINED}_{DATASET}'
print('参数就绪')

In [ ]:
# Cell 3 —— 拉仓库 + 装依赖（Colab 自带 torch，不要动它）
import os
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
%pip -q install "seisbench>=0.11" obspy h5py
os.environ['SEISBENCH_CACHE_ROOT'] = CACHE
print('仓库与依赖就绪')

In [ ]:
# Cell 4 —— 基座模型自检：一行白拿整个 DiTing（271 万条中国数据）的知识
import seisbench.models as sbm, torch
m = sbm.PhaseNet.from_pretrained(PRETRAINED)
print('标签顺序:', m.labels, '| 参数量:', sum(p.numel() for p in m.parameters()))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '无(去 修改>笔记本设置 开 T4)')

# 可选加餐：USTC-Pickers 省级 picker（可作 --init-weights 起点）
if not os.path.isdir('/content/USTC-Pickers'):
    !git clone -q --depth 1 https://github.com/JUNZHU-SEIS/USTC-Pickers.git /content/USTC-Pickers
!ls /content/USTC-Pickers/model_list/v0.1/ 2>/dev/null | head -40

In [ ]:
# Cell 5 —— 取标注数据（分块下载→抽窗→删块；断连后重跑本 Cell 自动续传）
# 若 /content 被清空但 Drive 有镜像，先恢复再续传
import os, shutil
for f in ('pool.hdf5', 'pool.hdf5.progress.json'):
    src, dst = f'{DRIVE}/{f}', f'/content/{f}'
    if not os.path.exists(dst) and os.path.exists(src):
        shutil.copy2(src, dst); print('已从 Drive 恢复', f)

!python -u {REPO_DIR}/scripts/chunked_fetch.py --dataset {DATASET} \
    --out {POOL} --cache {CACHE} --sr {SR} --win {WIN} --max-total {N_LABELED} \
    --mirror {DRIVE} --min-free-gb 8

In [ ]:
# Cell 6 —— 追加纯噪声窗（练“不误报”：数量罚 −0.5/个 是最狠的扣分项）
!python -u {REPO_DIR}/scripts/chunked_fetch.py --dataset {NOISE_SET} --noise \
    --out {POOL} --cache {CACHE} --sr {SR} --win {WIN} --max-total {N_LABELED + N_NOISE} \
    --mirror {DRIVE} --min-free-gb 8

In [ ]:
# Cell 7 —— 按事件切 train / holdout（防泄漏 + 零交集断言）
!python -u {REPO_DIR}/scripts/split_train_holdout.py --src {POOL} \
    --train /content/train.hdf5 --holdout /content/holdout.hdf5 --holdout-frac 0.1

In [ ]:
# Cell 8 —— 微调训练（checkpoint 全在 Drive；断连后重跑本 Cell 自动续训）
# best 由 holdout 官方计分守门：训不过基线就停在基线，绝不越训越差。
# --sr 必须与取数一致（脚本会与基座原生采样率双向校验，diting=50Hz）。
# 想从省级 picker 起步：加 --init-weights /content/USTC-Pickers/model_list/v0.1/<省>.pt
!python -u {REPO_DIR}/scripts/finetune_phasenet.py \
    --pretrained {PRETRAINED} --sr {SR} --win {WIN} \
    --data /content/train.hdf5 --holdout /content/holdout.hdf5 \
    --out {RUN_DIR} --resume \
    --epochs 12 --batch 64 --lr 3e-5 --holdout-max 300

In [ ]:
# Cell 9 —— 收尾：确认产物 + 下一步指令
import json, os
print('==== Drive 产物 ====')
!ls -lh {RUN_DIR}
prog = f'{RUN_DIR}/progress.json'
if os.path.exists(prog):
    print(json.load(open(prog)))
print()
print('下一步（回到本地/云主机）：')
print('  1) 下载 best.pt，用去年真题验证是否真提分：')
print('     python scripts/ab_compare.py --input round2.zip \\')
print('         --a "--pretrained diting" --b "--pretrained diting --weights best.pt" \\')
print('         --answer-package round2.zip')
print('  2) 赢了才换到 API：python scripts/serve_api.py --pretrained diting --weights best.pt')
print('  3) best.pt 放进仓库 weights/ 并 push，保住成果。')